
# Proyecto RAG con Mistral AI — Fase 1

## Objetivo de esta notebook

En esta primera etapa del proyecto vamos a:

1. Cargar el archivo `ventas.csv`
2. Analizar la estructura del dataset
3. Comprender los tipos de datos
4. Detectar valores nulos
5. Normalizar y limpiar los datos
6. Preparar el dataset para futuras etapas RAG
7. Analizar qué tipo de modelo de Mistral podría adaptarse mejor

⚠️ IMPORTANTE:
Todavía NO construiremos el chatbot ni el sistema RAG completo.
Primero debemos comprender correctamente los datos.


# SECCIÓN 1 — Instalación de librerías

En esta sección instalamos las librerías necesarias para:
- análisis de datos
- conexión con Mistral
- embeddings
- visualización
- procesamiento NLP


In [ ]:

# Instalación de librerías principales

!pip install -q pandas
!pip install -q numpy
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q mistralai
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

# SECCIÓN 2 — Importaciones

Aquí importamos las librerías que utilizaremos durante el análisis.


In [ ]:

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
pd.set_option("display.max_columns", None)

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


# SECCIÓN 3 — Configuración segura de API KEY

Google Colab permite almacenar secretos de forma segura.

Pasos:
1. Abrir el panel lateral izquierdo
2. Ir a "Secrets"
3. Crear:
   - Nombre: MistralProyecto
   - Valor: tu API KEY real

Luego podremos acceder a ella sin escribirla directamente en el notebook.


In [ ]:

# Importamos userdata desde Google Colab

from google.colab import userdata

# Obtenemos la API KEY guardada en Secrets
api_key = userdata.get("MistralProyecto")

# Verificación simple
if api_key:
    print("API KEY cargada correctamente.")
else:
    print("No se encontró la API KEY.")


API KEY cargada correctamente.


# SECCIÓN 4 — Test de conexión con Mistral

Antes de trabajar con modelos LLM debemos verificar:
- autenticación
- conexión
- acceso a modelos

⚠️ En esta etapa todavía NO elegimos definitivamente el modelo.
Primero probaremos conexión básica.


In [ ]:
from mistralai.client import Mistral

# Creamos el cliente
client = Mistral(api_key=api_key)

print("Cliente Mistral inicializado correctamente.")

Cliente Mistral inicializado correctamente.


# SECCIÓN 5 — Carga del CSV

Ahora cargaremos el dataset de ventas.




In [ ]:
# SECCIÓN — Carga interactiva del CSV

# Importamos la utilidad para subir archivos en Colab
from google.colab import files

# Abrirá una ventana para seleccionar el archivo
uploaded = files.upload()

# Obtener el nombre del archivo subido
file_name = list(uploaded.keys())[0]

# Leer el CSV utilizando pandas, especificando la codificación
df = pd.read_csv(file_name, encoding='latin1')

# Mostrar las primeras filas
df.head()

Saving ventas.csv to ventas.csv


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


# SECCIÓN 6 — Comprensión inicial del dataset

Antes de limpiar o normalizar datos debemos comprender:
- tamaño
- columnas
- tipos de datos
- posibles problemas


In [ ]:

# Dimensiones del dataset

print("Filas y columnas:")
print(df.shape)


Filas y columnas:
(2823, 25)


In [ ]:

# Nombres de columnas

print("Columnas disponibles:")
print(df.columns.tolist())


Columnas disponibles:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']


In [ ]:

# Tipos de datos

df.dtypes


,0
ORDERNUMBER,int64
QUANTITYORDERED,int64
PRICEEACH,float64
ORDERLINENUMBER,int64
SALES,float64
ORDERDATE,object
STATUS,object
QTR_ID,int64
MONTH_ID,int64
YEAR_ID,int64


In [ ]:

# Información general del dataset

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   int64  
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   object 
 6   STATUS            2823 non-null   object 
 7   QTR_ID            2823 non-null   int64  
 8   MONTH_ID          2823 non-null   int64  
 9   YEAR_ID           2823 non-null   int64  
 10  PRODUCTLINE       2823 non-null   object 
 11  MSRP              2823 non-null   int64  
 12  PRODUCTCODE       2823 non-null   object 
 13  CUSTOMERNAME      2823 non-null   object 
 14  PHONE             2823 non-null   object 
 15  ADDRESSLINE1      2823 non-null   object 
 16  ADDRESSLINE2      302 non-null    object 


# SECCIÓN 7 — Valores nulos

Los valores nulos son extremadamente importantes porque:
- afectan embeddings
- afectan retrieval
- afectan búsquedas semánticas
- afectan respuestas del chatbot

Debemos detectarlos antes de continuar.


In [ ]:

# Cantidad de valores nulos por columna

nulls = df.isnull().sum()

nulls


,0
ORDERNUMBER,0
QUANTITYORDERED,0
PRICEEACH,0
ORDERLINENUMBER,0
SALES,0
ORDERDATE,0
STATUS,0
QTR_ID,0
MONTH_ID,0
YEAR_ID,0


# SECCIÓN 8 — Normalización del dataset

La normalización busca:
- uniformar formatos
- limpiar inconsistencias
- preparar los datos para RAG
- mejorar embeddings y retrieval



In [ ]:

# Crear copia de seguridad

df_clean = df.copy()


In [ ]:

# Convertir nombres de columnas a minúsculas

df_clean.columns = df_clean.columns.str.lower()

# Eliminar espacios extra en nombres de columnas
df_clean.columns = df_clean.columns.str.strip()

df_clean.columns


Index(['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber',
       'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id',
       'productline', 'msrp', 'productcode', 'customername', 'phone',
       'addressline1', 'addressline2', 'city', 'state', 'postalcode',
       'country', 'territory', 'contactlastname', 'contactfirstname',
       'dealsize'],
      dtype='object')

In [ ]:

# Limpiar espacios extra en columnas tipo texto

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print("Espacios extra eliminados.")


Espacios extra eliminados.


In [ ]:

# Convertir texto a minúsculas (opcional)

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].str.lower()

print("Texto normalizado a minúsculas.")


Texto normalizado a minúsculas.


In [ ]:

# Verificar duplicados

duplicates = df_clean.duplicated().sum()

print(f"Cantidad de filas duplicadas: {duplicates}")


Cantidad de filas duplicadas: 0


In [ ]:

# Eliminar duplicados si existen

df_clean = df_clean.drop_duplicates()

print("Duplicados eliminados.")


Duplicados eliminados.


# SECCIÓN 9 — Estadísticas básicas

Aquí analizamos:
- distribución
- tipos de columnas
- comportamiento general


In [ ]:

# Estadísticas numéricas

df_clean.describe()


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,qtr_id,month_id,year_id,msrp
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


In [ ]:

# Estadísticas de texto/categorías

df_clean.describe(include="object")


,orderdate,status,productline,productcode,customername,phone,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
count,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823,2823
unique,252,6,7,109,92,91,92,10,73,17,74,19,4,77,72,3
top,11/14/2003 0:00,shipped,classic cars,s18_3232,euro shopping channel,(91) 555 94 44,"c/ moralzarzal, 86",nan,madrid,nan,28034,usa,emea,freyre,diego,medium
freq,38,2617,967,52,259,259,259,2521,304,1486,259,1004,1407,259,259,1384


# SECCIÓN 10 — Longitud del dataset

La longitud y estructura del dataset influirán directamente en:
- chunking
- embeddings
- retrieval
- elección del modelo Mistral


In [ ]:

# Cantidad total de registros

print(f"Cantidad total de registros: {len(df_clean)}")


Cantidad total de registros: 2823


# SECCIÓN 11 — Evaluación preliminar del modelo Mistral

Primero debemos:
- comprender volumen de datos
- tipo de consultas
- complejidad semántica
- tamaño contextual

Sin embargo, podemos realizar una evaluación preliminar.



## Posibles escenarios

### Si el dataset es:
- pequeño y estructurado → modelos pequeños pueden bastar
- muy grande → necesitamos mejor manejo contextual
- muy textual → modelos con mejor reasoning
- altamente analítico → modelos más potentes

## Candidatos preliminares

### Mistral Small
Ventajas:
- rápido
- económico
- ideal para RAG básico
- bueno para Colab

### Ministral
Ventajas:
- extremadamente liviano
- rápido
- útil para pruebas iniciales

### Modelos más grandes
Ventajas:
- mejor reasoning
- mejor contexto
- más costosos
- mayor consumo


# SECCIÓN 12 — Primer contacto con Mistral Small

En esta sección realizaremos nuestra primera interacción real
con un modelo de Mistral AI.

IMPORTANTE:
Todavía NO construiremos el sistema RAG completo.

Primero queremos comprender:

1. Cómo conectarnos al modelo
2. Cómo enviar prompts
3. Cómo recibir respuestas
4. Cómo funciona el flujo básico de un LLM

Más adelante integraremos:
- embeddings
- retrieval
- FAISS
- LangChain RAG
- agentes

Por ahora:
CSV + Prompt + Mistral

In [ ]:
# Importamos el cliente oficial de Mistral
from mistralai.client import Mistral

# Creamos el cliente utilizando nuestra API KEY
client = Mistral(api_key=api_key)

print("Cliente Mistral configurado correctamente.")

Cliente Mistral configurado correctamente.


# Test simple de generación de texto

Aquí realizaremos una prueba básica para verificar:

- conexión
- autenticación
- funcionamiento del modelo
- generación de respuestas

Usaremos:
mistral-small-latest


In [ ]:
# Enviar un mensaje simple al modelo

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {
            "role": "user",
            "content": "Explain briefly what a CSV dataset is."
        }
    ]
)

# Mostrar respuesta del modelo
print(response.choices[0].message.content)

A **CSV (Comma-Separated Values)** dataset is a plain text file format used to store tabular data (rows and columns) where each line represents a record, and values within a line are separated by commas (or other delimiters like semicolons or tabs).

### Key Features:
- **Simple & Human-Readable**: Easy to create, edit, and share.
- **No Complex Structure**: Unlike JSON or XML, it lacks nested data.
- **Common Use**: Widely used for spreadsheets (Excel), databases, and data exchange.
- **Example**:
  ```
  Name,Age,City
  Alice,25,New York
  Bob,30,Los Angeles
  ```

### Advantages:
- Lightweight and fast to process.
- Supported by most data tools (Pandas, Excel, SQL databases).

### Limitations:
- No support for complex data types (e.g., nested objects).
- Delimiter conflicts if data contains commas (requires escaping or quotes).

CSV is ideal for flat, structured datasets but may need preprocessing for messy or hierarchical data.
